# LSTM

## Import libraries

In [22]:
import ccxt
from datetime import datetime
import pandas as pd
import numpy as np

## Constants

In [23]:
CLIENT = ccxt.binance()
TICKER = "BTC/USDT"
END_DATE = datetime.now().timestamp() * 1000
START_DATE = CLIENT.parse8601("2010-11-03T20:00:00Z")
TIMEFRAME = "1d"
LAGS_NUM = 240
TRAIN_TEST_SPLIT_RATIO = 0.8
HIDDEN_NEURONS_NUM = 25
EPOCHS_NUM = 100
BATCH_SIZE = 32
DROPOUT_RATE = 0.1

In [24]:
TIMEFRAME_DICT = {
    "1m": 60 * 1000,
    "5m": 5 * 60 * 1000,
    "15m": 15 * 60 * 1000,
    "1h": 60 * 60 * 1000,
    "1d": 24 * 60 * 60 * 1000,
}

FREQ_DICT = {
    "1m": "T",
    "5m": "5T",
    "15m": "15T",
    "1h": "H",
    "1d": "D",
}

## Fetching data

In [25]:
def fetch_data(client, ticker, start_date, end_date, timeframe):
    data = []
    while start_date < end_date:
        data = data + client.fetch_ohlcv(
            ticker,
            timeframe=timeframe,
            since=start_date,
            limit=1000,
        )
        start_date = data[-1][0] + TIMEFRAME_DICT[timeframe]
    return data

def preprocess_data(data):
    data = pd.DataFrame(
        data,
        columns=["Date", "Open", "High", "Low", "Close", "Volume"]
    )
    data["Date"] = pd.to_datetime(data["Date"], unit="ms")
    data = data.set_index("Date")
    return data

In [26]:
data = fetch_data(CLIENT, TICKER, START_DATE, END_DATE, TIMEFRAME)
data = preprocess_data(data)
data.head()

,Open,High,Low,Close,Volume
Date,,,,,
2017-08-17,4261.48,4485.39,4200.74,4285.08,795.150377
2017-08-18,4285.08,4371.52,3938.77,4108.37,1199.888264
2017-08-19,4108.37,4184.69,3850.00,4139.98,381.309763
2017-08-20,4120.98,4211.08,4032.62,4086.29,467.083022
2017-08-21,4069.13,4119.62,3911.79,4016.00,691.743060


### Verifying duplicates

In [27]:
data.index.duplicated().sum()

0

### Verifying continuous of data

In [28]:
all_dates = pd.date_range(start=data.index.min(), end=data.index.max(), freq=FREQ_DICT[TIMEFRAME])
missing = all_dates.difference(data.index)
print(len(missing), missing)   # 0 => none missing dates

0 DatetimeIndex([], dtype='datetime64[ms]', freq='D')


## Processing data

In [29]:
data['log_return_close'] = np.log(data['Close'].shift(1) / data['Close'].shift(2))
data['log_return_open'] = np.log(data['Open'].shift(1) / data['Open'].shift(2))
data['log_return_high'] = np.log(data['High'].shift(1) / data['High'].shift(2))
data['log_return_low'] = np.log(data['Low'].shift(1) / data['Low'].shift(2))

In [30]:
data['log_spread_hl'] = np.log(data['High'].shift(1) / data['Low'].shift(1))
data['log_spread_co'] = np.log(data['Close'].shift(1) / data['Open'].shift(1))

In [ ]:
data['moving_average_return_close'] = data['log_return_close'].rolling(window=LAGS_NUM).mean()
data['deviation_return_close'] = data['log_return_close'].rolling(window=LAGS_NUM).std()

In [ ]:
data['volatility_ratio'] = data['log_return_close'].rolling(window=LAGS_NUM).std() / data['log_return_close'].rolling(window=LAGS_NUM).mean()

In [ ]:
data['true_range'] = np.max(data['High'].shift(1) - data['Low'].shift(1), np.abs(data['High'].shift(1) - data['Close'].shift(2)), np.abs(data['Low'].shift(1) - data['Close'].shift(2)))
data['average_true_range'] = data['true_range'].rolling(window=LAGS_NUM).mean()

In [ ]:
data['velocity'] = data['Close'].shift(1) - data['Close'].shift(2)
data['acceleration'] = data['Velocity'] - data['Velocity'].shift(1)

In [ ]:
data['volume_mean'] = data['Volume'].shift(1).rolling(window=LAGS_NUM).mean()
data['volume_std'] = data['Volume'].shift(1).rolling(window=LAGS_NUM).std()
data['delta_volume'] = (data['Volume'].shift(1) - data['volume_mean']) / data['volume_std']

In [ ]:
data['return_close'] = (data['Close'] - data['Close'].shift(1)) / data['Close'].shift(1)

In [ ]:
data['return_close_binary'] = (data['return_close'] > 0).astype(int)

In [32]:
data.head()

,Open,High,Low,Close,Volume,log_return_close,log_return_open,log_return_high,log_return_low,log_spread_hl,log_spread_co,moving_average_return_close,deviation_return_close
Date,,,,,,,,,,,,,
2017-08-17,4261.48,4485.39,4200.74,4285.08,795.150377,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-08-18,4285.08,4371.52,3938.77,4108.37,1199.888264,NaN,NaN,NaN,NaN,0.065565,0.005523,NaN,NaN
2017-08-19,4108.37,4184.69,3850.00,4139.98,381.309763,-0.042113,0.005523,-0.025715,-0.064392,0.104242,-0.042113,NaN,NaN
2017-08-20,4120.98,4211.08,4032.62,4086.29,467.083022,0.007665,-0.042113,-0.043678,-0.022795,0.083359,0.007665,NaN,NaN
2017-08-21,4069.13,4119.62,3911.79,4016.00,691.743060,-0.013053,0.003065,0.006287,0.046343,0.043303,-0.008454,NaN,NaN
